# Step 2: Registration Research (02_Registration_Research)

This notebook researches the geometric alignment between the reference and inspection images.

### Why Registration is Challenging here:
- The images are shot with slight camera movements, perspective changes, and rotations.
- The Rubik's cube has repetitive geometry and colors, which causes ambiguous feature descriptors.
- The images are extremely high resolution (12 Megapixels), which makes ORB matching on micro-textures unstable.

### Our Solution: Multi-Scale Alignment
1. Resize images to `1000x750` for descriptor computation and matching. This increases the scale/context of the ORB patch (making it capture corner shape rather than plastic reflection texture).
2. Perform Brute-Force Hamming matching and sort to keep the top 15% matches.
3. Compute the Homography matrix $H_{1000}$ using RANSAC.
4. Mathematically scale the homography matrix back to $H_{4000}$ using scale matrix multiplication ($S^{-1} H_{1000} S$).
5. Warp the original full-scale inspection image.

In [ ]:
%load_ext autoreload
%autoreload 2
import cv2
import numpy as np
import matplotlib.pyplot as plt
import core_functions as cf

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Successful Alignment: Scrambled Cell Defect
We load `ref_scrambled.jpg` and `inspect_scrambled_cell_defect.jpg` (which has no layer rotation). Since it is a rigid cube, alignment should be highly accurate.

In [ ]:
ref_img = cf.load_image('data/scrambled_pipeline/ref_scrambled.jpg')
inspect_img = cf.load_image('data/scrambled_pipeline/inspect_scrambled_cell_defect.jpg')

# Align
aligned_img, h_matrix, matches, kp_ins, kp_ref, inliers = cf.align_images_orb(inspect_img, ref_img)

print(f"Alignment completed!")
print(f"  Good Matches (top 15%): {len(matches)}")
print(f"  RANSAC Inliers: {inliers}")
if h_matrix is not None:
    det = h_matrix[0,0] * h_matrix[1,1] - h_matrix[0,1] * h_matrix[1,0]
    print(f"  Homography 2x2 determinant: {det:.4f}")

### Visualizing Feature Matches
We draw lines between matching keypoints to visualize the mapping.

In [ ]:
cf.visualize_matches(inspect_img, kp_ins, ref_img, kp_ref, matches)

### Verifying Registration with Alpha Blending
We blend the reference image and the warped inspection image with a 50/50 ratio. If alignment is perfect, the cube edges and stickers will align cleanly with zero doubling/blurring.

In [ ]:
blended = cv2.addWeighted(ref_img, 0.5, aligned_img, 0.5, 0)
plt.imshow(blended)
plt.title("Overlay Verification: Alpha Blend of Reference and Aligned Inspection Cube")
plt.axis('off')
plt.show()

## 2. Unsuccessful / Distorted Alignment: Scrambled Rotated Layer
When a layer is rotated, the rigid body assumption is broken. RANSAC will struggle to find inliers, and the calculated homography will be heavily distorted (low inliers or determinant far from 1.0).

In [ ]:
inspect_rotated = cf.load_image('data/scrambled_pipeline/inspect_scrambled_rotated_layer_hard.jpg')

# Align
aligned_rot, h_matrix_rot, matches_rot, kp_ins_rot, kp_ref_rot, inliers_rot = cf.align_images_orb(inspect_rotated, ref_img)

print("Alignment metrics on Rotated Layer (Hard):")
print(f"  Good Matches (top 15%): {len(matches_rot)}")
print(f"  RANSAC Inliers: {inliers_rot}")
if h_matrix_rot is not None:
    det_rot = h_matrix_rot[0,0] * h_matrix_rot[1,1] - h_matrix_rot[0,1] * h_matrix_rot[1,0]
    print(f"  Homography 2x2 determinant: {det_rot:.4f} (Notice how this deviates heavily from 1.0!)")

## Conclusion
By analyzing the Homography properties:
1. **RANSAC Inlier Count:** Drops significantly on rotated layers.
2. **Homography 2x2 Determinant:** Deviates heavily from 1.0 or becomes negative when alignment tries to project non-rigid structures.

We use these mathematical criteria to immediately classify heavily misaligned layers as **Rotated Layer** defects before proceeding to pixel subtraction.